# 01 Baselines

This notebook compares the baseline PID controller against the hover-linearized LQR controller on hover and simple tracking tasks.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.configuration import load_json_config
from src.control import build_controller
from src.dynamics import VTOLDynamicsModel
from src.plotting import plot_comparison
from src.simulations.core import simulate_experiment
from src.simulations.trajectories import hover_trajectory, circle_trajectory
import numpy as np


In [ ]:
config = load_json_config(REPO_ROOT / "config" / "default_params.json")
model = VTOLDynamicsModel.from_config(config)
initial_state = np.zeros(12)
initial_state[2] = -2.0
dt = config["simulations"]["dt"]
duration = config["simulations"]["duration_s"]

pid = build_controller("pid", model, config)
lqr = build_controller("lqr", model, config)

results_pid = simulate_experiment(model, pid, None, hover_trajectory(position=(0.0, 0.0, -2.0)), initial_state, duration, dt)
results_lqr = simulate_experiment(model, lqr, None, hover_trajectory(position=(0.0, 0.0, -2.0)), initial_state, duration, dt)
plot_comparison(results_pid, results_lqr, title="Hover baseline comparison")


In [ ]:
results_pid_track = simulate_experiment(model, build_controller("pid", model, config), None, circle_trajectory(period=duration), initial_state, duration, dt)
results_lqr_track = simulate_experiment(model, build_controller("lqr", model, config), None, circle_trajectory(period=duration), initial_state, duration, dt)
plot_comparison(results_pid_track, results_lqr_track, title="Circle tracking comparison")


**Research note:** In the current surrogate model, LQR usually gives smoother state regulation near hover, while PID remains a useful baseline because it reflects a familiar flight-control structure and is easy to retune when the vehicle model changes.
